# Rich Dict-First Construction

This notebook builds the same kind of rich engineering example as notebook 3, but keeps the user code dict-first.

The idea is:

- build the detailed specification record piece by piece with plain Python dictionaries
- derive the public `CellType` from that record
- build `CellInstance`, `Test`, and `Dataset` objects
- publish and reload the resulting `battinfo.publish.jsonld`


In [ ]:
import tempfile
from pathlib import Path
from pprint import pprint

from battinfo import CellInstance, Dataset, Test, derive_cell_type, load_publication, publish

workspace = Path(tempfile.mkdtemp(prefix="battinfo-rich-dict-api-"))
dataset_dir = workspace / "a123-anr26650m1-b-dict-demo"
raw_dir = dataset_dir / "timeseries" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
(raw_dir / "run-a.csv").write_text("time,voltage\n0,3.3\n1,3.2\n", encoding="utf-8")

print("Workspace:", workspace)
print("Dataset dir:", dataset_dir)


## Build The Rich Specification Record Piece By Piece

In [ ]:
positive_components = {
    "active_material": [
        {
            "name": "LFP",
            "property": {
                "mass_fraction": {"value": 0.94, "unit": "1"},
                "particle_d50": {"value": 1.8, "unit": "um"},
            },
        }
    ],
    "binder": [
        {
            "name": "PVDF",
            "property": {"mass_fraction": {"value": 0.03, "unit": "1"}},
        }
    ],
    "additive": [
        {
            "name": "Carbon black",
            "property": {"mass_fraction": {"value": 0.03, "unit": "1"}},
        },
        {
            "name": "Al2O3 coating particles",
            "property": {"mass_fraction": {"value": 0.005, "unit": "1"}},
        },
    ],
}

positive_electrode = {
    "coating": {
        "component": positive_components,
        "property": {
            "loading": {"value": 20.0, "unit": "mg/cm^2"},
            "porosity": {"value": 0.35, "unit": "1"},
            "thickness": {"value": 66.0, "unit": "um"},
        },
        "comment": "Representative production coating composition.",
    },
    "current_collector": {
        "name": "Al foil",
        "property": {
            "thickness": {"value": 15.0, "unit": "um"},
            "areal_mass": {"value": 40.5, "unit": "mg/cm^2"},
        },
        "comment": "Etched aluminum collector.",
    },
    "property": {"compaction_density": {"value": 2.9, "unit": "g/cm^3"}},
    "comment": "Positive electrode parameters from internal recipe.",
}

negative_electrode = {
    "coating": {
        "component": {
            "active_material": [
                {
                    "name": "Graphite",
                    "property": {"mass_fraction": {"value": 0.95, "unit": "1"}},
                }
            ],
            "binder": [
                {
                    "name": "CMC/SBR",
                    "property": {"mass_fraction": {"value": 0.04, "unit": "1"}},
                }
            ],
            "additive": [
                {
                    "name": "Carbon black",
                    "property": {"mass_fraction": {"value": 0.01, "unit": "1"}},
                }
            ],
        },
        "property": {
            "loading": {"value": 11.2, "unit": "mg/cm^2"},
            "thickness": {"value": 78.0, "unit": "um"},
        },
        "comment": "Representative negative coating.",
    },
    "current_collector": {
        "name": "Cu foil",
        "property": {"thickness": {"value": 10.0, "unit": "um"}},
    },
    "comment": "Negative electrode with graphite-dominant active material.",
}

electrolyte = {
    "family": "organic",
    "solvent_mixture": {
        "component": [
            {"name": "EC", "property": {"volume_fraction": {"value": 0.3, "unit": "1"}}},
            {"name": "EMC", "property": {"volume_fraction": {"value": 0.7, "unit": "1"}}},
        ],
        "property": {"water_content": {"max_value": 20, "unit": "ppm"}},
        "comment": "Base solvent system.",
    },
    "salt": {
        "name": "LiPF6",
        "cation": "Li+",
        "anion": "PF6-",
        "property": {"concentration": {"value": 1.0, "unit": "mol/L"}},
        "comment": "Standard 1M salt concentration.",
    },
    "additive": [
        {"name": "VC", "property": {"volume_fraction": {"value": 0.02, "unit": "1"}}},
        {"name": "FEC", "property": {"volume_fraction": {"value": 0.03, "unit": "1"}}},
    ],
    "property": {
        "conductivity": {"value": 10.5, "unit": "mS/cm"},
        "viscosity": {"value": 2.4, "unit": "mPa*s"},
    },
    "comment": "Electrolyte composition at 25 C.",
}

separator = {
    "material": "PE/PP trilayer",
    "property": {
        "thickness": {"value": 20.0, "unit": "um"},
        "porosity": {"value": 0.42, "unit": "1"},
    },
    "comment": "Microporous polyolefin separator.",
}


In [ ]:
cell_specification_record = {
    "schema_version": "1.0.0",
    "specification": {
        "id": "https://w3id.org/battinfo/cell-type/9qfb-4wrn-ynwc-ayjw",
        "manufacturer": "A123",
        "model": "ANR26650M1-B",
        "format": "cylindrical",
        "chemistry": "Li-ion",
        "positive_electrode_basis": "LFP",
        "negative_electrode_basis": "graphite",
        "size_code": "R26650",
        "property": {
            "nominal_capacity": {"typical_value": 2.5, "unit": "Ah"},
            "nominal_voltage": {"value": 3.3, "unit": "V"},
            "continuous_discharging_current": {"max_value": 50.0, "unit": "A"},
            "diameter": {"value": 26.0, "unit": "mm"},
            "height": {"value": 65.2, "unit": "mm"},
            "mass": {"value": 76.0, "unit": "g"},
        },
        "positive_electrode": positive_electrode,
        "negative_electrode": negative_electrode,
        "electrolyte": electrolyte,
        "separator": separator,
        "comment": [
            "Primary review artifact for BattINFO/BDF battery descriptor structure.",
            "Values are representative and intended for schema/mapping review.",
        ],
    },
    "provenance": {
        "source_type": "measurement",
        "source_name": "Catenaro 2021 ingestion pilot",
        "source_file": "ddata/a123/anr26650m1-b/catenaro-2021/battery.json",
        "source_url": "https://doi.org/10.17632/kxsbr4x3j2.2",
        "retrieved_at": 1772556000,
        "workflow_version": "battinfo-ingest-0.2.0",
        "comment": "Manually curated example promoted into the reusable BattINFO cell-type library.",
    },
    "comment": [
        "Reusable BattINFO library example for a curated commercial cylindrical cell type.",
        "Promoted from the canonical A123 review descriptor.",
    ],
}

pprint(cell_specification_record)


## Derive The Lightweight Public CellType

In [ ]:
cell_type = derive_cell_type(cell_specification_record)

pprint(cell_type.model_dump(exclude_none=True))


## Build The Publication Chain

In [ ]:
cell = CellInstance(
    cell_type=cell_type,
    serial_number="A123-ANR26650M1-B-DICT-DEMO-001",
)

test = Test(
    cell=cell,
    kind="cycle_life",
    protocol="0.5C baseline cycling",
    instrument="Arbin cycler",
    status="completed",
)

dataset = Dataset(
    path=str(dataset_dir),
    cell=cell,
    test=test,
    name="A123 ANR26650M1-B cycling dataset",
    description="Synthetic demo dataset for a rich dict-first notebook.",
)

pprint({
    "cell_type": cell_type.model_dump(exclude_none=True),
    "cell_instance": cell.model_dump(exclude_none=True),
    "test": test.model_dump(exclude_none=True),
    "dataset": dataset.model_dump(exclude_none=True),
})


## Publish And Reload

In [ ]:
report = publish(
    cell_specification=cell_specification_record,
    cell_type=cell_type,
    cell_instance=cell,
    test=test,
    dataset=dataset,
)

bundle = load_publication(dataset_dir / "battinfo.publish.jsonld")

pprint(report)
pprint({
    "cell_specification_id": bundle.cell_specification.id if bundle.cell_specification else None,
    "cell_type_id": bundle.cell_type.id,
    "cell_instance_id": bundle.cell_instance.id,
    "test_protocol": bundle.test.protocol.name,
    "test_instrument": bundle.test.instrument,
    "dataset_access_url": bundle.dataset.access_url,
})
